# Introduction
In this notebook we will demonstrate some useful ways LLM's can be employed beyond simple question and answering tasks. We will show how to use LLMs to:

+ write api calls to trigger other software (tool calls)
+ break down multi-step problems into multiple smaller steps (goal-decomposition)
+ categorize an input to a set of pre-defined labels.

Finally, we will combine these features to create a simple LLM agent capable of autonomously tackling multi-step problems.

# Starting the LLM Backend

We will run our LLM backend locally on the same node that we're running this notebook. There are many open-source backends available, for this notebook we will use ollama (https://ollama.com/) which has an extensive model library that can be found here (https://ollama.com/search).  For our agent we will use the 3 billion parameter version of the llama3.2 model. Note that our agent requires a model capable of making tool calls which can be identified in the ollama model library with the small tag that says "tool" under the model description.

### Start the ollama server in the background

In [2]:
import subprocess
import threading
import time

def run_ollama():
    subprocess.run("ollama serve", shell=True)

ollama_thread = threading.Thread(target=run_ollama)
ollama_thread.start()

# Give Ollama some time to start up
time.sleep(10)

2025/04/30 10:10:58 routes.go:1232: INFO server config env="map[CUDA_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:2048 OLLAMA_DEBUG:false OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_INTEL_GPU:false OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/home1/10156/gj3385/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_SCHED_SPREAD:false ROCR_VISIBLE_DEVICES: http_proxy

### Download the llama3.2 model

In [2]:
def download_model():
    subprocess.run("ollama run llama3.2", shell=True)

model_download_thread = threading.Thread(target=download_model)
model_download_thread.start()

[GIN] 2025/04/30 - 10:03:21 | 200 |    5.591394ms |       127.0.0.1 | HEAD     "/"


time=2025-04-30T10:03:21.304-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:03:21.333-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:03:21.371-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
⠙ 

[GIN] 2025/04/30 - 10:03:21 | 200 |  290.798068ms |       127.0.0.1 | POST     "/api/show"


⠹ ⠸ ⠼ ⠴ time=2025-04-30T10:03:21.876-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:03:21.899-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:03:21.900-05:00 level=WARN source=ggml.go:152 msg="key not found" key=llama.vision.block_count default=0
time=2025-04-30T10:03:21.900-05:00 level=INFO source=sched.go:722 msg="new model will fit in available VRAM in single GPU, loading" model=/home1/10156/gj3385/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff gpu=GPU-a7a202b9-f938-633f-07b5-48298401aad8 parallel=4 available=16775512064 required="3.7 GiB"
⠦ ⠦ ⠇ ⠇ time=2025-04-30T10:03:22.309-05:00 level=INFO source=server.go:105 msg="system memory" total="125.6 GiB" free="120.2 GiB" free_swap="0 B"
time=2025-04-30T10:03:22.309-05:00 level=WARN source=ggml.go:152 msg="key not found" key=llama.vision.block_count default=0
time=2025-04-3

⠹ llama_model_load_from_file_impl: using device CUDA0 (Quadro RTX 5000) - 15998 MiB free
⠼ llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from /home1/10156/gj3385/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str

[GIN] 2025/04/30 - 10:03:27 | 200 |  6.486592489s |       127.0.0.1 | POST     "/api/generate"


### Check that our model is available on the ollama server

This command lists the available local models our ollama backend has downloaded.  You should see an entry for the llama3.2 model we just downloaded

In [3]:
! ollama list

[GIN] 2025/04/30 - 10:11:12 | 200 |    6.503811ms |       127.0.0.1 | HEAD     "/"
[GIN] 2025/04/30 - 10:11:12 | 200 |    5.537873ms |       127.0.0.1 | GET      "/api/tags"
NAME               ID              SIZE      MODIFIED     
starcoder2:3b      9f4ae0aff61e    1.7 GB    5 weeks ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    2 months ago    


## Close ollama server
ONLY RUN THIS IF YOU WANT TO STOP THE OLLAMA SERVER!

In [1]:
! kill $(pgrep ollama)

kill: usage: kill [-s sigspec | -n signum | -sigspec] pid | jobspec ... or kill -l [sigspec]


# Define Software Tools for our LLM Agent

In order for our AI agent to actually _do_ something useful, it needs to be able to execute code on its own. In this case we will give our agent the capability of executing a few python functions that perform internet searches, basic math operations, look up the weather, and get the current time. The process for providing this functionality to the agent is simple, we will convert the source code (including the docstring) of each of our functions into a short text description that will be sent with our prompt to the LLM. The LLM will respond with our function name and the input arguments for the function which we can then execute.

### Functions we will use as our agent tools

We have 4 functions already provided in our agent codebase: 
1) get_duckduckgo_result()
2) do_math()
3) get_current_time()
4) get_current_weather()
   
Notice in the source code for **do_math()** that is copied below includes a docstring to define its purpose and inputs. These docstrings are important for the LLM to understand what our function does.

In [4]:
def do_math(a:int, op:str, b:int)->list:
    """
    Performs math on the inputs
    a: The first operand
    op: The operation to perform (one of '+', '-', '*', '/')
    b: The second operand
    """
    res = "Nan"
    if op == "+":
        res = str(int(a) + int(b))
    elif op == "-":
        res = str(int(a) - int(b))
    elif op == "*":
        res = str(int(a) * int(b))
    elif op == "/":
        if int(b) != 0:
            res = str(int(a) / int(b))
    return res

### How we transcribe functions into a string for the LLM to read

The function **generate_function_description()** converts a function's python code into a dictionary object that contains a description of what the function does and specifies its inputs and outputs so that the LLM will understand how use it. This function is copied from here https://github.com/meirm/ollama-tools. We can use any python function we want as a tool. You can create a list of tools to pass to the LLM like so:

    tools = [generate_function_description(<function 1 name>),
             generate_function_description(<function 2 name>),
             ...
             ]

#### Example
Here is an example that creates a list of tool objects using two of our built in functions. We will visualize what the LLM sees when we send it these tool descriptions by printing them with some light json formatting.

In [5]:
from agent_codebase.tools import generate_function_description, get_duckduckgo_result, do_math
import json

tools = [
    generate_function_description(get_duckduckgo_result),
    generate_function_description(do_math)]

print(f"Stringified tool descriptions:\n{json.dumps(tools, indent=4)}")

Stringified tool descriptions:
[
    {
        "type": "function",
        "function": {
            "name": "get_duckduckgo_result",
            "description": "Search for information online for the given query using DuckDuckGo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "str",
                        "description": "The search query to send to DuckDuckGo."
                    }
                },
                "required": [
                    "query"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "do_math",
            "description": "Performs math on the inputs",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "int",
                        "description": "The first operand"
           

### Prompting the LLM to use our tools

When querying the LLM to write a tool call we will use the function **llm_prompt_tool(prompt, tools)**.  This function takes as input our text prompt that describes the task we want the LLM to accomplish and the list of the tool objects created by **generate_function_description()**. 

    function_output = llm_prompt_tool(prompt='your prompt', tools=tools)

This will:
1) send the LLM your prompt and list of tools
2) the LLM will pick the function it wants to execute and respond with its input parameters
3) we then execute the chosen function with the generated input parameters and return the result as _function_output_

# Programatically Querying the LLM

LLMs by default only consume and generate text strings. In order to use an LLM programtically to generate other types of data objects like lists, numbers etc. we have to describe to the LLM the data structure we want it to generate and then parse the text response it gives us into a python object. We've created four functions that each prompt an LLM to generate a specific type of data.

**llm_prompt(prompt) -> str**

This function returns the response as a string.

**llm_prompt_tool(prompt, tools) -> str**

This function sends the prompt and a list of tools to the LLM, executes the tool call the LLM generates and returns the output of the tool function. Note, it only executes the first tool call the LLM wants to make.

**llm_create_list(prompt) -> dict**

This function queries the LLM with a text prompt asks it to return a dictionary object that contains a list with names and descriptions for each list item.

**llm_pick_option(prompt, options) -> int**

This function queries the LLM asking it to respond with a choice between any of the provided options.

# LLM Generation Examples

### Example: Basic text generation

The following example sends the *user_prompt* to our llm and then prints its response to the console.  The default model is set to be llama3.2, if you'd like to change it, you can add the optional input argument model="model name" to the **llm_prompt()** function like:

    response_text = llm_prompt(user_prompt,model="qwen:0.5b")

Note that you'll have to download any new models first, so for now let's stick with the llama3.2 model we already downloaded.

In [6]:
from agent_codebase.llm_functions import llm_prompt

# user prompt
user_prompt = f"Write a haiku about how awesome FFTs are."

response_text = llm_prompt(user_prompt)

print(f"LLM Response:\n{response_text}\n")

time=2025-04-30T10:04:09.018-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:11 | 200 |  2.489325629s |       127.0.0.1 | POST     "/api/chat"
LLM Response:
Fast and efficient
Solutions unfold with ease
Math's subtle delight



### Example: Tool calling

This example shows how to prompt the LLM to perform a tool call to get the current weather in a city.

In [7]:
from agent_codebase.llm_functions import llm_prompt_tool
from agent_codebase.tools import generate_function_description, get_current_weather

# build our tools list
tools = [generate_function_description(get_current_weather)]

# user prompt
user_prompt = "What's the weather in Austin Texas?"

response_text = llm_prompt_tool(user_prompt, tools)

print(f"LLM Response:\n{response_text}\n")

time=2025-04-30T10:04:18.287-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:18 | 200 |  483.568149ms |       127.0.0.1 | POST     "/api/chat"
Tool call:
function=Function(name='get_current_weather', arguments={'city': 'Austin'})

LLM Response:
The current temperature in Austin is: 24°C



### Example: Labeling

This example shows how to have the LLM make a choice of which input label to apply to the data in our prompt. Our prompt is a math task, and our labels will be the descriptions of each of our tool functions. Behind the scenes is a prompt template that is asking the LLM "Which of the {labels} is most appropriate for the {input_prompt}?"

The *none_option=True* means that a response of 0 will mean the LLM chose "none of the above", and then numbers 1-N correspond to a choice of each of our N tools. Our *user_prompt* is clearly outlining a math problem, so we should expect the LLM to chose option 3 which corresponds to our **do_math** function. To be good scientists, we'll have the model chose several times to see if it's option choice varies.

In [8]:
from agent_codebase.tools import generate_function_description, get_current_weather, get_current_time, do_math, get_duckduckgo_result
from agent_codebase.llm_functions import llm_pick_option

# user prompt
user_prompt = "Multiply 626183 with 182731"

# Create a list of tool objects that contain descriptions of our functions
tools = [
    generate_function_description(get_current_weather),
    generate_function_description(get_current_time),
    generate_function_description(do_math),
    generate_function_description(get_duckduckgo_result)
]       

# For each tool, pull out the dict object that contains the function name, description, and parameter description
func_descriptions = [func['function'] for func in tools]

# Send the llm our prompt 3 times, asking it to chose which function is most appropriate for our user_prompt
choices = [llm_pick_option(user_prompt, func_descriptions, none_option=True, show_prompt=False) for i in range(0,3)]

# print our results
labels = ['None of these'] + [d['name'] for d in func_descriptions]
for i in range(len(choices)):
    print(f"The LLM chose option {labels[choices[i]]}")


time=2025-04-30T10:04:23.928-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:24 | 200 |  305.281249ms |       127.0.0.1 | POST     "/api/chat"


time=2025-04-30T10:04:24.140-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:24 | 200 |  616.599851ms |       127.0.0.1 | POST     "/api/chat"


time=2025-04-30T10:04:24.757-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


...

time=2025-04-30T10:04:32.940-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:04:33.109-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:33 | 200 |   127.22574ms |       127.0.0.1 | POST     "/api/chat"


time=2025-04-30T10:04:33.240-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:33 | 200 |  350.806418ms |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/04/30 - 10:04:33 | 200 |   127.02188ms |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/04/30 - 10:04:33 | 200 |   53.774157ms |       127.0.0.1 | POST     "/api/chat"
The LLM chose option do_math
The LLM chose option do_math
The LLM chose option do_math


time=2025-04-30T10:04:33.612-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:04:33.719-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


### Example: Generate a list

In this example we'll generate a list which contains a step by step plan to accomplish a goal. Performing this type of goal decomposition allows agents to accomplish multi-step tasks atonomously. Try changing the goal in the *user_prompt* string yourself to see what the LLM is capable of planning for!

In [9]:
from agent_codebase.llm_functions import llm_create_list

# prompt for the model
user_prompt = """Break down the the following goal into subgoals

Goal: Create a step by step plan to perform perform basic calibration on all-sky survey data."""

# send the prompt to the LLM and have it return a list
# The list should be a dict object with fields:
#    {'list_description': 'Describe list contents here',
#     'content':[{'name': 'Name of list item 1', 'description', 'Description of list item 1'},...]}
generated_list = llm_create_list(user_prompt)

# Check if list is empty, if not, print out the names of the list items
if bool(generated_list):
    # pull out list names
    list_names = [item['name'] for item in generated_list['content']]
    
    # print list to console for viewing
    name_desc_array = [f"{i+1}. {step['name']} - {step['description']}" for i, step in enumerate(generated_list['content'])]
    name_desc_string = "\n".join(name_desc_array)
    print(f"Generated List:\n{name_desc_string}\n")
else:
    print(f"No List generated :(")

trying to create list [1/10] times...

time=2025-04-30T10:04:42.526-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:04:46 | 200 |  4.187044436s |       127.0.0.1 | POST     "/api/chat"
success! :D

Generated List:
1. Subgoal 1: Data Cleaning and Preparation - Ensure the dataset is free from errors, missing values, and outliers
2. Subgoal 2: Data Reduction and Processing - Apply necessary reductions (e.g., astrometry, radiometric corrections) to prepare data for calibration
3. Subgoal 3: Calibration Model Selection - Choose an appropriate calibration model suitable for the survey data (e.g., source-based, sky-based)
4. Subgoal 4: Source Detection and Identification - Detect and identify point sources within the survey data
5. Subgoal 5: Sky Map Construction - Create a detailed map of the sky with accurate positions of celestial objects
6. Subgoal 6: Radiometric Calibration - Calibrate the instrument's sensitivity to quantify its absolute flux measurement
7. Subgoal 7: Astrometric Calibration - Refine the astrometric solution for precise position measurements
8. Subgoal 8: Error 

### Example: Retrieval Augmented Generation (RAG)

Most large language models are not familiar with scientific jargon or subfield domain knowledge. Retrieval augmented generation (RAG) offers a way to programatically build a "cheat sheet" for LLMs so that they can answer questions on topics they were not trained on. In this example we will put the abstracts from two scientific papers as well as our schedule for today's tutorial into our RAG database.

In [10]:
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ------------------ Build RAG Database ------------------
# Let's build a database of relevant information we want our llm to draw from

# abstract text from an astronomy paper (Jeong-Eun Lee et al. 2023 ApJ 953 82)
abstract1 = "Most stars form in multiple-star systems. For a better understanding of their formation processes, it is important to resolve the individual protostellar components and the surrounding envelope and disk material at the earliest possible formation epoch, because the formation history can be lost in a few orbital timescales. Here we present Atacama Large Millimeter/submillimeter Array observational results of a young multiple protostellar system, IRAS 04239+2436, where three well-developed large spiral arms were detected in the shocked SO emission. Along the most conspicuous arm, the accretion streamer was also detected in the SO2 emission. The observational results are complemented by numerical magnetohydrodynamic simulations, where those large arms only appear in magnetically weakened clouds. Numerical simulations also suggest that the large triple spiral arms are the result of gravitational interactions between compact triple protostars and the turbulent infalling envelope."

# abstract text from a laser sail paper (Gabriel R. Jaffe et al. Nano Lett. 2023, 23, 15, 6852–6858)
abstract2 = "Laser sails propelled by gigawatt-scale ground-based laser arrays have the potential to reach relativistic speeds, traversing the solar system in hours and reaching nearby stars in years. Here, we describe the danger interplanetary dust poses to the survival of a laser sail during its acceleration phase. We show through multiphysics simulations how localized heating from a single optically absorbing dust particle on the sail can initiate a thermal runaway process that rapidly spreads and destroys the entire sail. We explore potential mitigation strategies, including increasing the in-plane thermal conductivity of the sail to reduce the peak temperature at hot spots and isolating the absorptive regions of the sail that can burn away individually."

# the schedule for today's tutorial
day4schedule = "Morning Session: 9am - 11:45pm, Lunch: 11:45pm - 1pm, Afternoon Sessions will run from 1pm - 4pm"

# Prepare our text data into llama_index Document objects
abstract_list = [abstract1, abstract2, day4schedule]
documents = [Document(text=text) for text in abstract_list]

# Create an indexed vector database of our documents
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Create retriever for top 1 document
retriever = index.as_retriever(similarity_top_k=1)
# ----------------------------------------------------------

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Now we will prompt the LLM normally with a question it definitely won't know the answer to, and then we'll do a generation using our RAG database where we retrieve the most relevant entry from our database, append that to our prompt and ask the LLM the same question again.

In [12]:
from agent_codebase.llm_functions import llm_prompt

# Our scientific question for the LLM
prompt = "How many arms does IRAS 04239+2436 have?"
# prompt = "How do I protect my laser sail against zodiacal dust? Respond with 1-2 sentences."
# prompt = "when's lunch?"

# =========== Generation without RAG ============
# send our question to the LLM
print(f"LLM Response no RAG:\n{llm_prompt(prompt)}")
# ===============================================

# =========== Generation with RAG ===============
print(f"\n============================================================")

# ------------------ Document Retrieval ------------------
# Retrieve whatever document is most relevant to our prompt
retrieved_nodes = retriever.retrieve(prompt)

# Extract document text
retrieved_docs = [node.text for node in retrieved_nodes]
print("\nText retrieved from database:", retrieved_docs)
# ----------------------------------------------------------

# --- Augment our LLM prompt with our retrieved document text ----
# let's embed the abstract text at the beginning of our prompt
document_text = "\n".join(retrieved_docs)
augmented_prompt = f"*Background Context*\n{document_text}\n\n*Query*\n{prompt}"

# send our augmented prompt to the LLM
print(f"\nLLM Response with RAG:\n{llm_prompt(augmented_prompt)}")


time=2025-04-30T10:06:58.618-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:06:59 | 200 |  933.975749ms |       127.0.0.1 | POST     "/api/chat"
LLM Response no RAG:
I couldn't find any information on "IRAS 04239+2436" that mentions the number of arms it has. It appears to be a reference to an astronomical object, possibly a star or galaxy, given the notation (RA, Dec) format.

Could you provide more context or clarify what IRAS 04239+2436 refers to?


Text retrieved from database: ['Most stars form in multiple-star systems. For a better understanding of their formation processes, it is important to resolve the individual protostellar components and the surrounding envelope and disk material at the earliest possible formation epoch, because the formation history can be lost in a few orbital timescales. Here we present Atacama Large Millimeter/submillimeter Array observational results of a young multiple protostellar system, IRAS 04239+2436, where three well-developed large spiral arms were detected in the shocked SO emission. Along the mo

time=2025-04-30T10:06:59.660-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:06:59 | 200 |  381.291578ms |       127.0.0.1 | POST     "/api/chat"

LLM Response with RAG:
According to the text, IRAS 04239+2436 has three well-developed large spiral arms in its shocked SO emission.


# LLM Agent Demo


Now we're ready to combine the functionality we've developed to create a simple LLM agent!

Usage: enter the goal for the AI Agent in the *user_prompt* and the agent will
1) Create a step by step plan to achieve it based on the provided *tools*
2) Execute each step sequentially by performing a tool call
3) Concatenate the output of all previous steps and perform a final LLM generation to complete the task

Note that the agent will always write a tool call for every step, even if none of the tools available are applicable. In this case there isn't a tool for "write poem" so the agent will likely do something silly like **do_math** for a step that requires text generation.  Try changing the *user_prompt* to see what kinds of tasks the agent can handle! You can also define your own functions in the cell below and add them to the tools list to make them available to the agent.

In [4]:
from agent_codebase.tools import generate_function_description, get_current_weather, get_current_time, do_math, get_duckduckgo_result
from agent_codebase.agent import run_agent

# the task for our agent
user_prompt = ("Look up the current temperature where the worlds two largest telescopes are based, "
               "add the two temperatures together. "
               "Search online for gear recommendations at this temperature and "
               "then write an epic poem about a graduate student journeying to the telescope "
               " at that temperature at the behest of their PhD advisor in the style of Homer.")

# create a list of the available tools
tools = [
    generate_function_description(get_current_weather),
    generate_function_description(get_current_time),
    generate_function_description(do_math),
    generate_function_description(get_duckduckgo_result),
]

# run the llm agent
run_agent(user_prompt, tools)


User Prompt: Look up the current temperature where the worlds two largest telescopes are based, add the two temperatures together. Search online for gear recommendations at this temperature and then write an epic poem about a graduate student journeying to the telescope  at that temperature at the behest of their PhD advisor in the style of Homer.

Tools the LLM has access to:
get_current_weather
get_current_time
do_math
get_duckduckgo_result

Generating step by step plan...
trying to create list [1/10] times...

time=2025-04-30T10:11:21.022-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:11:21.549-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:11:21.572-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32
time=2025-04-30T10:11:21.572-05:00 level=WARN source=ggml.go:152 msg="key not found" key=llama.vision.block_count default=0
time=2025-04-30T10:11:21.573-05:00 level=INFO source=sched.go:722 msg="new model will fit in available VRAM in single GPU, loading" model=/home1/10156/gj3385/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff gpu=GPU-a7a202b9-f938-633f-07b5-48298401aad8 parallel=4 available=16775512064 required="3.7 GiB"
time=2025-04-30T10:11:21.975-05:00 level=INFO source=server.go:105 msg="system memory" total="125.6 GiB" free="120.2 GiB" free_swap="0 B"
time=2025-04-30T10:11:21.975-05:00 l

llama_model_loader: - kv  25:                      tokenizer.ggml.merges arr[str,280147]  = ["Ġ Ġ", "Ġ ĠĠĠ", "ĠĠ ĠĠ", "...
llama_model_loader: - kv  26:                tokenizer.ggml.bos_token_id u32              = 128000
llama_model_loader: - kv  27:                tokenizer.ggml.eos_token_id u32              = 128009
llama_model_loader: - kv  28:                    tokenizer.chat_template str              = {{- bos_token }}\n{%- if custom_tools ...
llama_model_loader: - kv  29:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   58 tensors
llama_model_loader: - type q4_K:  168 tensors
llama_model_loader: - type q6_K:   29 tensors
print_info: file format = GGUF V3 (latest)
print_info: file type   = Q4_K - Medium
print_info: file size   = 1.87 GiB (5.01 BPW) 
load: special tokens cache size = 256
load: token to piece cache size = 0.7999 MB
print_info: arch             = llama
print_info: vocab_only       = 0
print_info: n_ctx_train      = 

success! :D
[GIN] 2025/04/30 - 10:11:32 | 200 | 11.250360554s |       127.0.0.1 | POST     "/api/chat"

Generated Step by Step plan:
1. Step 1: Get current temperature at Mauna Kea, Hawaii (home of the world's largest telescope)
2. Step 2: Parse temperature from weather output
3. Step 3: Get current temperature at Atacama Large Millimeter/submillimeter Array (ALMA), Chile (home of the second largest telescope)
4. Step 4: Parse temperature from weather output
5. Step 5: Add two temperatures together
6. Step 6: Get gear recommendations for average temperature
7. Step 7: Write an epic poem about graduate student journey to telescope

Executing step [1/7] Step 1: Get current temperature at Mauna Kea, Hawaii (home of the world's largest telescope)



time=2025-04-30T10:11:33.100-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


...

[GIN] 2025/04/30 - 10:11:38 | 200 |  426.539314ms |       127.0.0.1 | POST     "/api/chat"
Tool call:
function=Function(name='get_duckduckgo_result', arguments={'query': 'gear for astronomy at 22°F'})

tool_results: It should also be as "fast" as possible. This is indicated by a low f-number, like f/1.8 or f/2.8. The lower this number, the better it will be at capturing light under dark skies. See the Best Lenses for Astrophotography. The tripod should be sturdy to minimize any wobble that will ruin long exposure images.
Executing step [7/7] Step 7: Write an epic poem about graduate student journey to telescope



time=2025-04-30T10:11:39.719-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:11:40 | 200 |  368.904603ms |       127.0.0.1 | POST     "/api/chat"
Tool call:
function=Function(name='get_duckduckgo_result', arguments={'query': 'gear recommendations for 25°c'})

tool_results: The riding season is in full swing by the time the weather warms up to the 65-70°F ( 18-21°C) range. And breaking out the first short-sleeve jersey of the season is a great reward for slogging ...


time=2025-04-30T10:11:41.060-05:00 level=WARN source=ggml.go:152 msg="key not found" key=general.alignment default=32


[GIN] 2025/04/30 - 10:11:45 | 200 |  4.582829254s |       127.0.0.1 | POST     "/api/chat"

====================== Final Prompt Template ======================

Goal: Look up the current temperature where the worlds two largest telescopes are based, add the two temperatures together. Search online for gear recommendations at this temperature and then write an epic poem about a graduate student journeying to the telescope  at that temperature at the behest of their PhD advisor in the style of Homer..

    Step by step plan:
    1. Step 1: Get current temperature at Mauna Kea, Hawaii (home of the world's largest telescope)
2. Step 2: Parse temperature from weather output
3. Step 3: Get current temperature at Atacama Large Millimeter/submillimeter Array (ALMA), Chile (home of the second largest telescope)
4. Step 4: Parse temperature from weather output
5. Step 5: Add two temperatures together
6. Step 6: Get gear recommendations for average temperature
7. Step 7: Write an epic poem about 